# Proposal smoke tests (after both shards complete)
Five gates:
1. **No-gold-leak:** Context does not contain gold action (similarity > 0.8)
2. **Qwen hit rate:** Count proposals equal to gold (exact or sim > 0.95)
3. **Per-option-type text baselines:** Bag-of-words classifier (gold vs proposal, gold vs real T1/T2/T3)
4. **Gold-memorisation check:** Proposals do not memorise gold text
5. **Per-source counts:** Real options by source and tier; proposal stats

In [ ]:
import json, gzip, os
from collections import Counter, defaultdict

# Load proposals from shards
proposals_by_id = {}
for shard in [0, 1]:
    path = f"/kaggle/working/out/proposals_shard{shard}.jsonl"
    if os.path.exists(path):
        with open(path) as f:
            for line in f:
                r = json.loads(line)
                proposals_by_id[r["decision_id"]] = r
        print(f"Loaded {len(proposals_by_id)} proposals from shard {shard}")
    else:
        print(f"Shard {shard} not yet available")

print(f"Total proposals: {len(proposals_by_id)}")

In [ ]:
# Load decisions (sample; real would load full 45,500)
print("\nLoading decisions...")
decisions = {}
input_dir = "/kaggle/input/laya-head-states-v1"
if os.path.exists(input_dir):
    for shard_file in sorted([f for f in os.listdir(input_dir) if f.startswith("pool_") and f.endswith(".jsonl")])[:1]:
        with open(os.path.join(input_dir, shard_file)) as f:
            for line in f:
                r = json.loads(line)
                decisions[r.get("decision_id", "")] = r
                if len(decisions) >= 100:
                    break
else:
    # Dummy
    decisions = {f"d{i}": {"task": f"Task {i}", "history": [], "options": [{"text": f"action {i}"}]} for i in range(10)}

print(f"Loaded {len(decisions)} decisions")

In [ ]:
# Gate 1: no-gold-leak (sample of 10)
print("\n=== Gate 1: No-gold-leak ===")
gold_leak_count = 0
for decision_id in list(decisions.keys())[:10]:
    dec = decisions[decision_id]
    gold = dec.get("options", [{}])[0].get("text", "")
    context = json.dumps({"task": dec.get("task", ""), "history": dec.get("history", [])})
    if gold and len(gold) > 20 and gold in context:
        gold_leak_count += 1
        print(f"LEAK: {decision_id}")

print(f"Result: {gold_leak_count} of {min(10, len(decisions))} decisions leaked gold (PASS if 0)")

In [ ]:
# Gate 2: Qwen hit rate
print("\n=== Gate 2: Qwen hit rate ===")
hits = 0
total_proposals = 0
for decision_id, prop in list(proposals_by_id.items())[:100]:
    if decision_id in decisions:
        gold = decisions[decision_id].get("options", [{}])[0].get("text", "").strip()
        if prop["proposal_text"].strip() == gold:
            hits += 1
        total_proposals += 1

if total_proposals > 0:
    hit_rate = 100 * hits / total_proposals
    print(f"Result: {hits}/{total_proposals} = {hit_rate:.1f}%")
else:
    print(f"Result: No proposals to check (expected in early smoke)")

In [ ]:
# Gate 3: Per-option-type text baselines
print("\n=== Gate 3: Per-option-type text baselines ===")
print("Smoke: length-based classifier (gold vs proposal).")

# Collect option lengths by type
option_lengths = {"gold": [], "proposal": []}
for decision_id in list(decisions.keys())[:50]:
    dec = decisions[decision_id]
    gold = dec.get("options", [{}])[0].get("text", "")
    if gold:
        option_lengths["gold"].append(len(gold))
    
    prop = proposals_by_id.get(decision_id, {})
    if prop.get("proposal_text"):
        option_lengths["proposal"].append(len(prop["proposal_text"]))

for opt_type in ["gold", "proposal"]:
    if option_lengths[opt_type]:
        avg_len = sum(option_lengths[opt_type]) / len(option_lengths[opt_type])
        print(f"  {opt_type:10s}: avg length {avg_len:.0f} chars")

print(f"\n  Full run: train TF-IDF on option text only. Flag if accuracy > 0.60.")

In [ ]:
# Gate 4: Gold-memorisation check
print("\n=== Gate 4: Gold-memorisation check ===")
print("Smoke: check if proposals contain significant substring of gold.")

memorise_count = 0
for decision_id in list(decisions.keys())[:10]:
    dec = decisions[decision_id]
    gold = dec.get("options", [{}])[0].get("text", "")
    prop = proposals_by_id.get(decision_id, {})
    
    if gold and prop.get("proposal_text") and len(gold) > 20:
        # Check if middle 70% of gold is in proposal
        start, end = int(len(gold) * 0.15), int(len(gold) * 0.85)
        gold_middle = gold[start:end]
        if gold_middle in prop["proposal_text"]:
            memorise_count += 1
            print(f"  MEMORISE: {decision_id}")

print(f"Result: {memorise_count} of {min(10, len(decisions))} memorized gold (PASS if 0)")

In [ ]:
# Gate 5: Per-source counts
print("\n=== Gate 5: Per-source counts ===")
print("Real options per source: (pending hybrid data load)")
print("Proposal stats:")
if proposals_by_id:
    print(f"  Total proposals: {len(proposals_by_id)}")
    avg_output_tokens = sum(p.get('output_tokens', 0) for p in proposals_by_id.values()) / len(proposals_by_id)
    avg_thinking_tokens = sum(p.get('thinking_tokens', 0) for p in proposals_by_id.values()) / len(proposals_by_id)
    print(f"  Mean output tokens: {avg_output_tokens:.0f}")
    print(f"  Mean thinking tokens: {avg_thinking_tokens:.0f}")
else:
    print(f"  (No proposals loaded in smoke)")

print(f"\nReal options (pending): per source (nebius, SWE-smith) and tier (T1, T2, T3)")
print(f"\n" + "="*60)
print("Smoke tests completed")
print("="*60)